# aaivaft: running example and replication of Acharya et al. (2026)

This notebook runs the `aaivaft` command, version 1.0.0, on the pediatric ICU data. It has three goals.

The first goal is to show how the command works, with the settings used in the Stata Journal paper.

The second goal is to check the command against Table 2 of Acharya, Khalaf, Menon, Voia, Yazbeck, and Wensley (2026), Canadian Journal of Economics (CJE). The notebook places the published numbers next to ours and reports the difference in each cell.

The third goal is to document the checks behind the command. Each check has its own section, so a reader can rerun it and see the evidence.

Contents

1. Setup
2. Files
3. The estimator in brief
4. Running example: PRISM and PIM
5. Calibrating the empirical quantile score Fo
6. Comparison with Table 2 of the CJE paper
7. Validation checks
8. Summary

The full notebook takes about 30 minutes to run. Most of the time goes to the frailty models and to the simulations with 5,000 draws.

## 1. Setup

Stata runs from Python through `pystata`. The first cell points Python to the Stata installation. The second cell sets the folders, makes sure Stata uses the `aaivaft` files in the `code` folder, and loads the data.

Change the three paths in the second cell to your own folders. The output folder must exist.

In [1]:
import stata_setup
stata_setup.config("C:/Program Files/StataNow19", "se")
from pystata import stata


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       SE—Standard Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user network, expiring 23 May 2027
Serial number: 401909309750
  Licensed to: Leonardo Sanchez-Aragon
               ESPOL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.

Running C:\Users\user\ado\personal\profile.do ...


In [2]:
%%stata
global code "path\to\aaivaft\code"
global data "path\to\aaivaft\data\HE_May2023_V13"
global out  "path\to\aaivaft\output"

adopath ++ "$code"
discard
which aaivaft
which aaivaft_export

* moremata is required for the rank computations
capture mata: mm_ranks((1\2\3), 1, 0)
if _rc {
    display as error "moremata is missing: ssc install moremata"
}
else {
    display as text "moremata found"
}

use "$data", clear
describe lnlosh PRISM PIM Trauma hsp1-hsp6 Agecat Cardiac Nchrnx PrevICU, short
summarize lnlosh PRISM PIM Trauma


. global code "path\to\aaaft\code"

. global data "path\to\aaaft\data\HE_May2023_V13"

. global out  "path\to\aaaft\output"

. 
. adopath ++ "$code"
  [1]              "path\to\aaaft\code"
  [2]  (BASE)      "C:\Program Files\StataNow19/ado\base/"
  [3]  (SITE)      "C:\Program Files\StataNow19/ado\site/"
  [4]              "."
  [5]  (PERSONAL)  "C:\Users\user\ado\personal/"
  [6]  (PLUS)      "C:\Users\user\ado\plus/"
  [7]  (OLDPLACE)  "c:\ado/"

. discard

. which aaaft
path\to\aaaft\code\aaaft.ado
*! version 1.0.0  19oct2026
*! aaaft: rank-based IV inference for accelerated failure time models
*! Authors: Sanchez-Aragon, Acharya, Menon, and Voia
*! Based on: Acharya, Khalaf, Menon, Voia, Yazbeck, and Wensley (2026), Canadia
> n Journal of Economics, forthcoming
*! Requires: moremata (ssc install moremata) and aaaft_export.ado

. which aaaft_export
path\to\aaaft\code\aaaft_export.ado
*! version 1.0.0  19oct2026
*! aaaft_export: writes aaaft results to a CSV file
*! Companion to aa


. if _rc {
.     display as error "moremata is missing: ssc install moremata"
. }

. else {
.     display as text "moremata found"
moremata found
. }

. 
. use "$data", clear

. describe lnlosh PRISM PIM Trauma hsp1-hsp6 Agecat Cardiac Nchrnx PrevICU, sho
> rt

Variable      Storage   Display    Value
    name         type    format    label      Variable label
-------------------------------------------------------------------------------
lnlosh          float   %9.0g                 
PRISM           byte    %10.0g                PRISM
PIM             double  %10.0g                PIM
Trauma          byte    %10.0g                Trauma
hsp1            byte    %10.0g                hsp1
hsp2            byte    %10.0g                hsp2
hsp3            byte    %10.0g                hsp3
hsp4            byte    %10.0g                hsp4
hsp5            byte    %10.0g                hsp5
hsp6            byte    %10.0g                hsp6
Agecat          byte    %10.0g                A

## 2. Files

The command is made of two files that must sit in the same folder on the ado path.

1. `aaivaft.ado` is the command.
2. `aaivaft_export.ado` writes the results to a CSV file. The command calls it through `export()`, and it can also run by itself after `aaivaft`.
3. `aaivaft.sthlp` is the help file.

Three do files reproduce the results of the paper.

1. `calibrate_fo.do` builds the draws of the error for the empirical quantile score, `fo_PRISM.dta` and `fo_PIM.dta`.
2. `running_example.do` runs the example of Section 6 of the paper and exports the results.
3. `robust_fo.do` produces the robustness table for the calibration of $F_o$.

This notebook runs the same steps in one place, plus the validation checks.

## 3. The estimator in brief

The AFT model for the log length of stay is

$$\ln T_i = \beta\, s_i + x_i'\gamma + \varepsilon_i ,$$

where $s_i$ is the severity score (PRISM or PIM), $x_i$ holds the exogenous covariates, and the instrument $z_i$ is the trauma indicator.

For each candidate value $\beta_0$ on a grid, the command partials out the covariates and ranks the residuals. A score function $c(\cdot)$ turns the ranks $R_i$ into scores $c\big(R_i/(n+1)\big)$. The rank statistic is

$$G(\beta_0) = \frac{\big(\tilde z' c\big)^2}{\tilde z'\tilde z},$$

where $\tilde z = M_X z$ is the instrument residualized on the covariates. The CJE paper writes the projection with $z$ itself. Any fixed version of the instrument keeps the test exact. The option `zcenter` uses the instrument centered at its mean instead, and Section 7.5 compares the two.

The command reports four sets of results.

1. The Hodges Lehmann Sen (HLS) point estimate, the grid value that minimizes $G$.
2. The rank confidence set, all grid values with $G(\beta_0) < g_{\text{calc}}(\alpha)$. Under the null, the ranks are a random permutation, so the critical value is simulated once from random permutations. This set is exact and robust to weak instruments.
3. The least squares (LS) IV point estimate, the value where the generalized Anderson Rubin statistic is zero.
4. The LS confidence set, obtained by inverting the generalized Anderson Rubin statistic. It is exact only if the errors follow the distribution of each score.

The scores are the logistic $\ln\big(u/(1-u)\big)$, the Gumbel or maximum extreme value $-\ln(-\ln u)$, the exponential $-\ln(1-u)$, and, with `fo()`, the empirical quantile score $F_o^{-1}(u)$.

## 4. Running example: PRISM and PIM

All runs use the same grid, from 0.01 to 0.60 with step 0.005. The simulations use 5,000 draws and seed 12345. With these settings the bounds of the logistic score move at most one grid step across seeds (see Section 7.3).

### 4.1 IV estimates with the three closed form scores

In [3]:
%%stata
global covs  hsp1-hsp6 Agecat Cardiac Nchrnx PrevICU
global grid  gridmin(0.01) gridmax(0.60) gridstep(0.005)
global sim   reps(5000) seed(12345)

use "$data", clear
aaivaft lnlosh PRISM $covs, instrument(Trauma) $grid $sim
estimates store prism_iv
scalar prism_ls_pt = e(beta_LS)
scalar prism_ls_lo = e(CI_lo_logistic)
scalar prism_ls_hi = e(CI_hi_logistic)


. global covs  hsp1-hsp6 Agecat Cardiac Nchrnx PrevICU

. global grid  gridmin(0.01) gridmax(0.60) gridstep(0.005)

. global sim   reps(5000) seed(12345)

. 
. use "$data", clear

. aaaft lnlosh PRISM $covs, instrument(Trauma) $grid $sim



------------------------------------------------------------
 Rank IV inference for AFT duration models (aaaft)
------------------------------------------------------------
 Dependent variable  : lnlosh
 Treatment variable  : PRISM
 Instrument          : Trauma
 Covariates          : hsp1 hsp2 hsp3 hsp4 hsp5 hsp6 Agecat Cardiac Nchrnx Prev
> ICU
 Observations        : 10044
 Beta grid           : .01 to .6 (step .005)
 Simulations (reps)  : 5000
 Confidence level    : 95%
------------------------------------------------------------

 Computing rank estimates...

 Computing least-squares IV confidence sets...

------------------------------------------------------------
  Results
------------------------------------------------------------

  Hodges-Lehmann-Sen point estimates (Rank IV):
  ------------------------------------------------
   Score                   Beta
  ------------------------------------------------
   Logistic                  0.2000
   Gumbel (Extreme Value)    0.

In [4]:
%%stata
aaivaft lnlosh PIM $covs, instrument(Trauma) $grid $sim
estimates store pim_iv
scalar pim_ls_pt = e(beta_LS)
scalar pim_ls_lo = e(CI_lo_logistic)
scalar pim_ls_hi = e(CI_hi_logistic)


. aaaft lnlosh PIM $covs, instrument(Trauma) $grid $sim

------------------------------------------------------------
 Rank IV inference for AFT duration models (aaaft)
------------------------------------------------------------
 Dependent variable  : lnlosh
 Treatment variable  : PIM
 Instrument          : Trauma
 Covariates          : hsp1 hsp2 hsp3 hsp4 hsp5 hsp6 Agecat Cardiac Nchrnx Prev
> ICU
 Observations        : 10044
 Beta grid           : .01 to .6 (step .005)
 Simulations (reps)  : 5000
 Confidence level    : 95%
------------------------------------------------------------

 Computing rank estimates...

 Computing least-squares IV confidence sets...

------------------------------------------------------------
  Results
------------------------------------------------------------

  Hodges-Lehmann-Sen point estimates (Rank IV):
  ------------------------------------------------
   Score                   Beta
  ------------------------------------------------
   Logistic 

### 4.2 Rank estimates without the instrument

Without `instrument()`, the command reports the rank only benchmark. It ignores the endogeneity of the severity score. The gap between the two estimates shows the direction and size of the endogeneity bias.

In [5]:
%%stata
aaivaft lnlosh PRISM $covs, $grid
estimates store prism_noiv
aaivaft lnlosh PIM $covs, $grid
estimates store pim_noiv

estimates table prism_iv prism_noiv pim_iv pim_noiv, b(%6.3f)

estimates restore prism_iv
scalar g_prism = e(HLS_logistic)
estimates restore prism_noiv
scalar g_prism = g_prism - e(HLS_logistic)
estimates restore pim_iv
scalar g_pim = e(HLS_logistic)
estimates restore pim_noiv
scalar g_pim = g_pim - e(HLS_logistic)
display as text "Gap IV minus no IV, logistic score:  PRISM " as result %6.3f g_prism ///
    as text "   PIM " as result %6.3f g_pim


. aaaft lnlosh PRISM $covs, $grid

------------------------------------------------------------
 Rank IV inference for AFT duration models (aaaft)
------------------------------------------------------------
 Dependent variable  : lnlosh
 Treatment variable  : PRISM
 Instrument          : (none — rank estimates only)
 Covariates          : hsp1 hsp2 hsp3 hsp4 hsp5 hsp6 Agecat Cardiac Nchrnx Prev
> ICU
 Observations        : 10044
 Beta grid           : .01 to .6 (step .005)
------------------------------------------------------------

 Computing rank estimates...

------------------------------------------------------------
  Results
------------------------------------------------------------

  Hodges-Lehmann-Sen point estimates (Rank, no IV):
  ------------------------------------------------
   Score                   Beta
  ------------------------------------------------
   Logistic                  0.0650
   Gumbel (Extreme Value)    0.0750
   Exponential               0.0800
 

The IV correction goes in opposite directions for the two scores. For PRISM the IV estimate is larger than the rank only benchmark. For PIM it is smaller.

## 5. Calibrating the empirical quantile score Fo

The central estimator of the CJE paper uses the empirical quantile score $c(u) = F_o^{-1}(u)$. The CJE paper says that $F_o$ comes from a simulated convolution distribution or a split sample, but it does not give the calibration. We rebuild $F_o$ from a Weibull model with Gamma frailty.

In that model the error of $\ln T$ is

$$\varepsilon = \sigma_g\,(W - \ln \alpha),$$

where $W$ is a standard minimum extreme value variable, $\alpha$ is a Gamma frailty with mean 1 and variance $\theta_g$, and $\sigma_g$ is the scale of hospital $g$. When $\theta = 1$ and the scale is common, $\varepsilon$ is exactly logistic.

The parameters come from the data. `streg` does not allow `ancillary()` together with `frailty()`, so the model is estimated separately in each hospital. The calibration ignores the endogeneity of the severity score, because only the shape of the error matters. Adding a constant to a score or scaling it does not change the confidence set.

The next cell defines two small programs. The first estimates $\sigma_g$, $\theta_g$, and the share of each hospital. The second simulates 100,000 draws of $\varepsilon$ and saves them. They are the same steps as in `calibrate_fo.do`.

In [6]:
%%stata
global B 100000

capture program drop calib_hosp
program define calib_hosp
    args sev cond
    quietly count `cond'
    local ntot = r(N)
    forvalues g = 1/6 {
        quietly streg `sev' Agecat Cardiac Nchrnx PrevICU `cond' & hsp`g' == 1, ///
            distribution(weibull) frailty(gamma) time
        scalar sig`g' = exp(-_b[/ln_p])
        scalar th`g'  = e(theta)
        scalar sh`g'  = e(N) / `ntot'
    }
end

capture program drop sim_fo
program define sim_fo
    args file
    preserve
    clear
    quietly set obs $B
    set seed 2026
    quietly gen double u = runiform()
    quietly gen byte g = 1
    local cum = 0
    forvalues k = 1/5 {
        local cum = `cum' + sh`k'
        quietly replace g = `k' + 1 if u > `cum'
    }
    quietly gen double sig = .
    quietly gen double th  = .
    forvalues k = 1/6 {
        quietly replace sig = sig`k' if g == `k'
        quietly replace th  = th`k'  if g == `k'
    }
    quietly gen double W     = ln(-ln(runiform()))
    quietly gen double alpha = cond(th < 1e-6, 1, rgamma(1/th, th))
    quietly gen double eps   = sig * (W - ln(alpha))
    keep eps
    quietly save `"`file'"', replace
    restore
end


. global B 100000

. 
. capture program drop calib_hosp

. program define calib_hosp
  1.     args sev cond
  2.     quietly count `cond'
  3.     local ntot = r(N)
  4.     forvalues g = 1/6 {
  5.         quietly streg `sev' Agecat Cardiac Nchrnx PrevICU `cond' & hsp`g' 
> == 1, ///
>             distribution(weibull) frailty(gamma) time
  6.         scalar sig`g' = exp(-_b[/ln_p])
  7.         scalar th`g'  = e(theta)
  8.         scalar sh`g'  = e(N) / `ntot'
  9.     }
 10. end

. 
. capture program drop sim_fo

. program define sim_fo
  1.     args file
  2.     preserve
  3.     clear
  4.     quietly set obs $B
  5.     set seed 2026
  6.     quietly gen double u = runiform()
  7.     quietly gen byte g = 1
  8.     local cum = 0
  9.     forvalues k = 1/5 {
 10.         local cum = `cum' + sh`k'
 11.         quietly replace g = `k' + 1 if u > `cum'
 12.     }
 13.     quietly gen double sig = .
 14.     quietly gen double th  = .
 15.     forvalues k = 1/6 {
 16.         quie

In [7]:
%%stata
foreach sev in PRISM PIM {
    use "$data", clear
    quietly gen double t = exp(lnlosh)
    quietly stset t
    calib_hosp `sev' "if 1"
    display as text _n "Calibration for `sev'"
    forvalues g = 1/6 {
        display as text "  hospital `g': sigma = " as result %6.4f sig`g' ///
            as text "   theta = " as result %6.4f th`g' ///
            as text "   share = " as result %5.3f sh`g'
    }
    sim_fo "$out\fo_`sev'.dta"
}


. foreach sev in PRISM PIM {
  2.     use "$data", clear
  3.     quietly gen double t = exp(lnlosh)
  4.     quietly stset t
  5.     calib_hosp `sev' "if 1"
  6.     display as text _n "Calibration for `sev'"
  7.     forvalues g = 1/6 {
  8.         display as text "  hospital `g': sigma = " as result %6.4f sig`g' 
> ///
>             as text "   theta = " as result %6.4f th`g' ///
>             as text "   share = " as result %5.3f sh`g'
  9.     }
 10.     sim_fo "$out\fo_`sev'.dta"
 11. }

Calibration for PRISM
  hospital 1: sigma = 0.4791   theta = 1.3347   share = 0.142
  hospital 2: sigma = 0.3624   theta = 2.2236   share = 0.052
  hospital 3: sigma = 0.3934   theta = 2.1719   share = 0.333
  hospital 4: sigma = 0.3465   theta = 2.5524   share = 0.090
  hospital 5: sigma = 0.4211   theta = 1.8340   share = 0.293
  hospital 6: sigma = 0.4501   theta = 1.6657   share = 0.090

Calibration for PIM
  hospital 1: sigma = 0.5017   theta = 1.2216   share = 0.142
  hospital 2: sigma =

All frailty variances are above 1. This gives the error a mild right skew, so $F_o$ lies between the logistic and the Gumbel scores.

### 5.1 aaivaft with the empirical quantile score

These are the central estimates of the paper. The `export()` option writes them to a CSV file, one row per score.

In [8]:
%%stata
use "$data", clear
aaivaft lnlosh PRISM $covs, instrument(Trauma) $grid $sim ///
    fo("$out\fo_PRISM.dta") export("$out\PRISM_iv.csv")
scalar prism_q_pt = e(HLS_empq)
scalar prism_q_lo = e(CIR_lo_empq)
scalar prism_q_hi = e(CIR_hi_empq)

aaivaft lnlosh PIM $covs, instrument(Trauma) $grid $sim ///
    fo("$out\fo_PIM.dta") export("$out\PIM_iv.csv")
scalar pim_q_pt = e(HLS_empq)
scalar pim_q_lo = e(CIR_lo_empq)
scalar pim_q_hi = e(CIR_hi_empq)


. use "$data", clear

. aaaft lnlosh PRISM $covs, instrument(Trauma) $grid $sim ///
>     fo("$out\fo_PRISM.dta") export("$out\PRISM_iv.csv")

------------------------------------------------------------
 Rank IV inference for AFT duration models (aaaft)
------------------------------------------------------------
 Dependent variable  : lnlosh
 Treatment variable  : PRISM
 Instrument          : Trauma
 Covariates          : hsp1 hsp2 hsp3 hsp4 hsp5 hsp6 Agecat Cardiac Nchrnx Prev
> ICU
 Observations        : 10044
 Beta grid           : .01 to .6 (step .005)
 Simulations (reps)  : 5000
 Confidence level    : 95%
------------------------------------------------------------

 Computing rank estimates...

 Computing least-squares IV confidence sets...

------------------------------------------------------------
  Results
------------------------------------------------------------

  Hodges-Lehmann-Sen point estimates (Rank IV):
  ------------------------------------------------
   Scor

In [9]:
import pandas as pd
out = r"path\to\aaivaft\output"
pd.read_csv(out + r"\PRISM_iv.csv").round(3)

,score,hls,rank_lo,rank_hi,ls_point,ls_lo,ls_hi,depvar,treatvar,instrument,N,level
0,logistic,0.200,0.075,0.360,0.16,0.088,0.262,lnlosh,PRISM,Trauma,10044,95
1,gumbel,0.260,0.140,0.470,0.16,0.089,0.262,lnlosh,PRISM,Trauma,10044,95
2,exponential,0.285,0.165,0.565,0.16,0.089,0.261,lnlosh,PRISM,Trauma,10044,95
3,empq,0.225,0.110,0.400,0.16,NaN,NaN,lnlosh,PRISM,Trauma,10044,95


## 6. Comparison with Table 2 of the CJE paper

Each row gives the lower bound, the point estimate, and the upper bound. The published values come from Table 2 of Acharya et al. (2026). The differences are ours minus the paper. For the LS rows we use the bounds simulated with logistic errors.

In [10]:
%%stata
clear
input str6 outcome str24 method p_lo p_pt p_hi
"PRISM" "Empirical quantile rank" 0.094 0.229 0.398
"PRISM" "Least squares IV"        0.088 0.169 0.262
"PIM"   "Empirical quantile rank" 0.061 0.125 0.190
"PIM"   "Least squares IV"        0.070 0.130 0.189
end

gen double o_lo = .
gen double o_pt = .
gen double o_hi = .
replace o_lo = prism_q_lo  in 1
replace o_pt = prism_q_pt  in 1
replace o_hi = prism_q_hi  in 1
replace o_lo = prism_ls_lo in 2
replace o_pt = prism_ls_pt in 2
replace o_hi = prism_ls_hi in 2
replace o_lo = pim_q_lo    in 3
replace o_pt = pim_q_pt    in 3
replace o_hi = pim_q_hi    in 3
replace o_lo = pim_ls_lo   in 4
replace o_pt = pim_ls_pt   in 4
replace o_hi = pim_ls_hi   in 4

gen double d_lo = o_lo - p_lo
gen double d_pt = o_pt - p_pt
gen double d_hi = o_hi - p_hi
format p_* o_* d_* %6.3f

list outcome method p_lo p_pt p_hi o_lo o_pt o_hi, noobs sep(2) abbrev(10)
list outcome method d_lo d_pt d_hi, noobs sep(2) abbrev(10)
export delimited using "$out\comparison_cje_table2.csv", replace


. clear

. input str6 outcome str24 method p_lo p_pt p_hi

       outcome                    method       p_lo       p_pt       p_hi
  1. "PRISM" "Empirical quantile rank" 0.094 0.229 0.398
  2. "PRISM" "Least squares IV"        0.088 0.169 0.262
  3. "PIM"   "Empirical quantile rank" 0.061 0.125 0.190
  4. "PIM"   "Least squares IV"        0.070 0.130 0.189
  5. end

. 
. gen double o_lo = .
(4 missing values generated)

. gen double o_pt = .
(4 missing values generated)

. gen double o_hi = .
(4 missing values generated)

. replace o_lo = prism_q_lo  in 1
(1 real change made)

. replace o_pt = prism_q_pt  in 1
(1 real change made)

. replace o_hi = prism_q_hi  in 1
(1 real change made)

. replace o_lo = prism_ls_lo in 2
(1 real change made)

. replace o_pt = prism_ls_pt in 2
(1 real change made)

. replace o_hi = prism_ls_hi in 2
(1 real change made)

. replace o_lo = pim_q_lo    in 3
(1 real change made)

. replace o_pt = pim_q_pt    in 3
(1 real change made)

. replace o_hi = pim_

In [11]:
# Same table as a pandas DataFrame, for a cleaner view
df = stata.pdataframe_from_data()
df.round(3)

,outcome,method,p_lo,p_pt,p_hi,o_lo,o_pt,o_hi,d_lo,d_pt,d_hi
0,PRISM,Empirical quantile rank,0.094,0.229,0.398,0.110,0.225,0.400,0.016,-0.004,0.002
1,PRISM,Least squares IV,0.088,0.169,0.262,0.088,0.160,0.262,0.000,-0.009,0.000
2,PIM,Empirical quantile rank,0.061,0.125,0.190,0.065,0.125,0.185,0.004,0.000,-0.005
3,PIM,Least squares IV,0.070,0.130,0.189,0.070,0.130,0.189,0.000,0.000,-0.000


### 6.1 Reference estimates without endogeneity correction

Table 2 of the CJE paper also reports likelihood estimates that ignore endogeneity: a Weibull AFT model and a Weibull AFT model with Gamma frailty. They are estimated outside `aaivaft`. The next cell estimates them with `streg` as a check on the data and the specification.

In [12]:
%%stata
use "$data", clear
quietly gen double t = exp(lnlosh)
quietly stset t

foreach sev in PRISM PIM {
    quietly streg `sev' Agecat Cardiac Nchrnx PrevICU hsp2-hsp6, ///
        distribution(weibull) time
    display as text "`sev' Weibull:       " as result %6.3f _b[`sev'] ///
        as text "  (" as result %6.3f _b[`sev'] - invnormal(0.975)*_se[`sev'] ///
        as text ", " as result %6.3f _b[`sev'] + invnormal(0.975)*_se[`sev'] as text ")"
    quietly streg `sev' Agecat Cardiac Nchrnx PrevICU hsp2-hsp6, ///
        distribution(weibull) frailty(gamma) time
    display as text "`sev' Gamma frailty: " as result %6.3f _b[`sev'] ///
        as text "  (" as result %6.3f _b[`sev'] - invnormal(0.975)*_se[`sev'] ///
        as text ", " as result %6.3f _b[`sev'] + invnormal(0.975)*_se[`sev'] as text ")"
}
display as text _n "CJE Table 2: PRISM Weibull 0.075 (0.071, 0.080), Gamma frailty 0.071 (0.068, 0.075)"
display as text    "             PIM   Weibull 0.338 (0.321, 0.356), Gamma frailty 0.256 (0.241, 0.270)"


. use "$data", clear

. quietly gen double t = exp(lnlosh)

. quietly stset t

. 
. foreach sev in PRISM PIM {
  2.     quietly streg `sev' Agecat Cardiac Nchrnx PrevICU hsp2-hsp6, ///
>         distribution(weibull) time
  3.     display as text "`sev' Weibull:       " as result %6.3f _b[`sev'] ///
>         as text "  (" as result %6.3f _b[`sev'] - invnormal(0.975)*_se[`sev']
>  ///
>         as text ", " as result %6.3f _b[`sev'] + invnormal(0.975)*_se[`sev'] 
> as text ")"
  4.     quietly streg `sev' Agecat Cardiac Nchrnx PrevICU hsp2-hsp6, ///
>         distribution(weibull) frailty(gamma) time
  5.     display as text "`sev' Gamma frailty: " as result %6.3f _b[`sev'] ///
>         as text "  (" as result %6.3f _b[`sev'] - invnormal(0.975)*_se[`sev']
>  ///
>         as text ", " as result %6.3f _b[`sev'] + invnormal(0.975)*_se[`sev'] 
> as text ")"
  6. }
PRISM Weibull:        0.075  ( 0.070,  0.080)
PRISM Gamma frailty:  0.071  ( 0.068,  0.075)
PIM Weibull:        0.338  ( 0.3

### 6.2 Reading the comparison

The LS confidence sets match the paper to the third decimal for both scores, and the LS point estimate matches for PIM. This confirms that the sample, the covariates, and the instrument are the same as in the paper.

The LS point estimate for PRISM is 0.160 against 0.169 in the paper. The LS routine does not use $F_o$, and the same code matches PIM exactly. The CJE paper does not explain this value.

The empirical quantile rank estimates are within one grid step of the paper. The confidence sets are close and a little narrower. The CJE paper does not give the calibration of $F_o$ or the number of permutations. Section 7.6 shows that the remaining gap is smaller than the change produced by moving the frailty variance within a plausible range. An exact match would also need the same seed and the same random number generator.

## 7. Validation checks

### 7.1 Logistic draws reproduce the logistic score

If $F_o$ is built from logistic draws, the empirical quantile score should give the same results as the logistic score. This checks the `fo()` mechanism.

In [13]:
%%stata
clear
set obs 100000
set seed 2026
gen double eps = logit(runiform())
save "$out\fo_logistic.dta", replace

use "$data", clear
aaivaft lnlosh PRISM $covs, instrument(Trauma) $grid $sim nols fo("$out\fo_logistic.dta")


. clear

. set obs 100000
Number of observations (_N) was 0, now 100,000.

. set seed 2026

. gen double eps = logit(runiform())

. save "$out\fo_logistic.dta", replace
file path\to\aaaft\output\fo_logistic.dta saved

. 
. use "$data", clear

. aaaft lnlosh PRISM $covs, instrument(Trauma) $grid $sim nols fo("$out\fo_logi
> stic.dta")

------------------------------------------------------------
 Rank IV inference for AFT duration models (aaaft)
------------------------------------------------------------
 Dependent variable  : lnlosh
 Treatment variable  : PRISM
 Instrument          : Trauma
 Covariates          : hsp1 hsp2 hsp3 hsp4 hsp5 hsp6 Agecat Cardiac Nchrnx Prev
> ICU
 Observations        : 10044
 Beta grid           : .01 to .6 (step .005)
 Simulations (reps)  : 5000
 Confidence level    : 95%
------------------------------------------------------------

 Computing rank estimates...

------------------------------------------------------------
  Results
----------------------

The empirical quantile score built from logistic draws matches the logistic score within one grid step. The point estimate is 0.205 against 0.200, and the set is (0.080, 0.360) against (0.075, 0.360). The small gap comes from the extreme quantiles, which are estimated from few draws.

### 7.2 Results do not depend on the sort order of the data

The command adds a tiny jitter, $10^{-12}$ times the observation index, to break ties in the residuals. The simulations also use a fixed order of the instrument. The next cell runs PRISM with the data in six different orders. All rows should be identical.

In [14]:
%%stata
use "$data", clear
tempfile res
postfile h order_id hls_l hls_e cr_lo_l cr_hi_l ls_lo_l ls_hi_l using `res'
forvalues k = 0/5 {
    if `k' == 0 {
        sort PRISM lnlosh
    }
    else {
        set seed `=1000 + `k''
        gen double u = runiform()
        sort u
        drop u
    }
    quietly aaivaft lnlosh PRISM $covs, instrument(Trauma) $grid reps(1000) seed(12345)
    post h (`k') (e(HLS_logistic)) (e(HLS_exponential)) ///
        (e(CIR_lo_logistic)) (e(CIR_hi_logistic)) ///
        (e(CI_lo_logistic)) (e(CI_hi_logistic))
}
postclose h
use `res', clear
list, noobs sep(0)


. use "$data", clear

. tempfile res

. postfile h order_id hls_l hls_e cr_lo_l cr_hi_l ls_lo_l ls_hi_l using `res'

. forvalues k = 0/5 {
  2.     if `k' == 0 {
  3.         sort PRISM lnlosh
  4.     }
  5.     else {
  6.         set seed `=1000 + `k''
  7.         gen double u = runiform()
  8.         sort u
  9.         drop u
 10.     }
 11.     quietly aaaft lnlosh PRISM $covs, instrument(Trauma) $grid reps(1000) 
> seed(12345)
 12.     post h (`k') (e(HLS_logistic)) (e(HLS_exponential)) ///
>         (e(CIR_lo_logistic)) (e(CIR_hi_logistic)) ///
>         (e(CI_lo_logistic)) (e(CI_hi_logistic))
 13. }

. postclose h

. use `res', clear

. list, noobs sep(0)

  +--------------------------------------------------------------------+
  | order_id   hls_l   hls_e   cr_lo_l   cr_hi_l    ls_lo_l    ls_hi_l |
  |--------------------------------------------------------------------|
  |        0      .2    .285      .075      .365   .0851128   .2686387 |
  |        1      .2    .285   

### 7.3 Monte Carlo error of the confidence bounds

The bounds depend on simulated critical values. The next cell runs PRISM with five seeds, first with 1,000 draws and then with 5,000 draws.

In [15]:
%%stata
use "$data", clear
tempfile res
postfile h reps seed_id cr_lo_l cr_hi_l cr_lo_e cr_hi_e ls_lo_l ls_hi_l using `res'
foreach R in 1000 5000 {
    forvalues s = 1/5 {
        quietly aaivaft lnlosh PRISM $covs, instrument(Trauma) $grid reps(`R') seed(`s')
        post h (`R') (`s') (e(CIR_lo_logistic)) (e(CIR_hi_logistic)) ///
            (e(CIR_lo_exponential)) (e(CIR_hi_exponential)) ///
            (e(CI_lo_logistic)) (e(CI_hi_logistic))
    }
}
postclose h
use `res', clear
tabstat cr_lo_l cr_hi_l cr_lo_e cr_hi_e ls_lo_l ls_hi_l, by(reps) stat(min max sd) format(%6.4f)


. use "$data", clear

. tempfile res

. postfile h reps seed_id cr_lo_l cr_hi_l cr_lo_e cr_hi_e ls_lo_l ls_hi_l using
>  `res'

. foreach R in 1000 5000 {
  2.     forvalues s = 1/5 {
  3.         quietly aaaft lnlosh PRISM $covs, instrument(Trauma) $grid reps(`R
> ') seed(`s')
  4.         post h (`R') (`s') (e(CIR_lo_logistic)) (e(CIR_hi_logistic)) ///
>             (e(CIR_lo_exponential)) (e(CIR_hi_exponential)) ///
>             (e(CI_lo_logistic)) (e(CI_hi_logistic))
  5.     }
  6. }

. postclose h

. use `res', clear

. tabstat cr_lo_l cr_hi_l cr_lo_e cr_hi_e ls_lo_l ls_hi_l, by(reps) stat(min ma
> x sd) format(%6.4f)

Summary statistics: Min, Max, SD
Group variable: reps 

    reps |   cr_lo_l   cr_hi_l   cr_lo_e   cr_hi_e   ls_lo_l   ls_hi_l
---------+------------------------------------------------------------
    1000 |    0.0700    0.3500    0.1650    0.5300    0.0858    0.2585
         |    0.0800    0.3700    0.1750    0.5650    0.0901    0.2672
         |    0.0045    0

With 5,000 draws the logistic bounds move at most one grid step and the LS bounds less than 0.003. With 1,000 draws the logistic upper bound moves up to four steps. The upper bound of the exponential score moves more, up to six steps even with 5,000 draws, because $G$ is flat in that region. This is why the running example uses `reps(5000)` and reports the seed.

### 7.4 Orientation of the asymmetric scores

The logistic score is symmetric. The Gumbel and exponential scores are not, so the direction of their skew matters. Changing the sign of the outcome and of the treatment keeps $\beta$ but reflects the residuals. The Gumbel score then becomes the minimum extreme value score of a Weibull model.

In [16]:
%%stata
use "$data", clear
gen double n_lnlosh = -lnlosh
gen double n_PRISM  = -PRISM
gen double n_PIM    = -PIM
aaivaft n_lnlosh n_PRISM $covs, instrument(Trauma) $grid $sim nols
aaivaft n_lnlosh n_PIM   $covs, instrument(Trauma) $grid $sim nols


. use "$data", clear

. gen double n_lnlosh = -lnlosh

. gen double n_PRISM  = -PRISM

. gen double n_PIM    = -PIM

. aaaft n_lnlosh n_PRISM $covs, instrument(Trauma) $grid $sim nols

------------------------------------------------------------
 Rank IV inference for AFT duration models (aaaft)
------------------------------------------------------------
 Dependent variable  : n_lnlosh
 Treatment variable  : n_PRISM
 Instrument          : Trauma
 Covariates          : hsp1 hsp2 hsp3 hsp4 hsp5 hsp6 Agecat Cardiac Nchrnx Prev
> ICU
 Observations        : 10044
 Beta grid           : .01 to .6 (step .005)
 Simulations (reps)  : 5000
 Confidence level    : 95%
------------------------------------------------------------

 Computing rank estimates...

------------------------------------------------------------
  Results
------------------------------------------------------------

  Hodges-Lehmann-Sen point estimates (Rank IV):
  ------------------------------------------------
   Score 

The logistic results do not change. The reflected scores move away from the paper, and the paper lies between the logistic and the original Gumbel scores. So $F_o$ has a mild right skew, which agrees with the calibration in Section 5.

### 7.5 Residualized versus centered instrument

By default the rank statistic uses the instrument residualized on the covariates. The option `zcenter` uses the instrument centered at its mean. Both keep the rank set exact, but they give different statistics when the covariates explain part of the instrument. In the ICU data they explain about 9% of the variance of the trauma indicator.

In [17]:
%%stata
use "$data", clear
quietly regress Trauma $covs
display as text "R-squared of Trauma on the covariates: " as result %5.3f e(r2)

tempfile res
postfile h str6 sev str12 z hls_q lo_q hi_q using `res'
foreach sev in PRISM PIM {
    quietly aaivaft lnlosh `sev' $covs, instrument(Trauma) $grid $sim nols ///
        fo("$out\fo_`sev'.dta")
    post h ("`sev'") ("residualized") (e(HLS_empq)) (e(CIR_lo_empq)) (e(CIR_hi_empq))
    quietly aaivaft lnlosh `sev' $covs, instrument(Trauma) $grid $sim nols ///
        fo("$out\fo_`sev'.dta") zcenter
    post h ("`sev'") ("centered") (e(HLS_empq)) (e(CIR_lo_empq)) (e(CIR_hi_empq))
}
postclose h
use `res', clear
format hls_q lo_q hi_q %6.3f
list, noobs sepby(sev)
display as text "CJE Table 2: PRISM 0.229 (0.094, 0.398), PIM 0.125 (0.061, 0.190)"


. use "$data", clear

. quietly regress Trauma $covs

. display as text "R-squared of Trauma on the covariates: " as result %5.3f e(r
> 2)
R-squared of Trauma on the covariates: 0.088

. 
. tempfile res

. postfile h str6 sev str12 z hls_q lo_q hi_q using `res'

. foreach sev in PRISM PIM {
  2.     quietly aaaft lnlosh `sev' $covs, instrument(Trauma) $grid $sim nols /
> //
>         fo("$out\fo_`sev'.dta")
  3.     post h ("`sev'") ("residualized") (e(HLS_empq)) (e(CIR_lo_empq)) (e(CI
> R_hi_empq))
  4.     quietly aaaft lnlosh `sev' $covs, instrument(Trauma) $grid $sim nols /
> //
>         fo("$out\fo_`sev'.dta") zcenter
  5.     post h ("`sev'") ("centered") (e(HLS_empq)) (e(CIR_lo_empq)) (e(CIR_hi
> _empq))
  6. }

. postclose h

. use `res', clear

. format hls_q lo_q hi_q %6.3f

. list, noobs sepby(sev)

  +----------------------------------------------+
  |   sev              z   hls_q    lo_q    hi_q |
  |----------------------------------------------|
  | PRISM   residualize

The residualized instrument brings the results closer to the CJE paper. For PRISM the upper bound moves from 0.370 to 0.400, against 0.398 in the paper. For PIM the point estimate matches exactly and the lower bound gets closer. Only the PIM upper bound is closer with the centered instrument. This is why the residualized instrument is the default.

### 7.6 Robustness of the Fo calibration

The next cell repeats the empirical quantile estimates under five calibrations: per hospital (base), a single pooled model, $\theta = 1$, and the frailty variance doubled and halved. A sixth version calibrates on one half of the sample and estimates on the other. This split keeps the exactness of the rank set. The same steps are in `robust_fo.do`.

In [18]:
%%stata
tempfile res
postfile h str6 sev str8 calib hls_q lo_q hi_q hls_l lo_l hi_l using `res'

foreach sev in PRISM PIM {
    use "$data", clear
    quietly gen double t = exp(lnlosh)
    quietly stset t

    calib_hosp `sev' "if 1"
    forvalues g = 1/6 {
        scalar bth`g' = th`g'
    }
    sim_fo "$out\rob_`sev'_base.dta"
    forvalues g = 1/6 {
        scalar th`g' = 1
    }
    sim_fo "$out\rob_`sev'_theta1.dta"
    forvalues g = 1/6 {
        scalar th`g' = 2 * bth`g'
    }
    sim_fo "$out\rob_`sev'_thx2.dta"
    forvalues g = 1/6 {
        scalar th`g' = 0.5 * bth`g'
    }
    sim_fo "$out\rob_`sev'_thx05.dta"

    quietly streg `sev' Agecat Cardiac Nchrnx PrevICU hsp2-hsp6, ///
        distribution(weibull) frailty(gamma) time
    forvalues g = 1/6 {
        scalar sig`g' = exp(-_b[/ln_p])
        scalar th`g'  = e(theta)
    }
    sim_fo "$out\rob_`sev'_pooled.dta"

    foreach c in base pooled theta1 thx2 thx05 {
        quietly aaivaft lnlosh `sev' $covs, instrument(Trauma) $grid $sim nols ///
            fo("$out\rob_`sev'_`c'.dta")
        post h ("`sev'") ("`c'") (e(HLS_empq)) (e(CIR_lo_empq)) (e(CIR_hi_empq)) ///
            (e(HLS_logistic)) (e(CIR_lo_logistic)) (e(CIR_hi_logistic))
    }

    set seed 777
    gen byte half = runiform() < 0.5
    calib_hosp `sev' "if half == 1"
    sim_fo "$out\rob_`sev'_split.dta"
    quietly aaivaft lnlosh `sev' $covs if half == 0, instrument(Trauma) $grid $sim nols ///
        fo("$out\rob_`sev'_split.dta")
    post h ("`sev'") ("split") (e(HLS_empq)) (e(CIR_lo_empq)) (e(CIR_hi_empq)) ///
        (e(HLS_logistic)) (e(CIR_lo_logistic)) (e(CIR_hi_logistic))
}
postclose h
use `res', clear
format hls_q lo_q hi_q hls_l lo_l hi_l %6.3f
list, noobs sepby(sev)


. tempfile res

. postfile h str6 sev str8 calib hls_q lo_q hi_q hls_l lo_l hi_l using `res'

. 
. foreach sev in PRISM PIM {
  2.     use "$data", clear
  3.     quietly gen double t = exp(lnlosh)
  4.     quietly stset t
  5. 
.     calib_hosp `sev' "if 1"
  6.     forvalues g = 1/6 {
  7.         scalar bth`g' = th`g'
  8.     }
  9.     sim_fo "$out\rob_`sev'_base.dta"
 10.     forvalues g = 1/6 {
 11.         scalar th`g' = 1
 12.     }
 13.     sim_fo "$out\rob_`sev'_theta1.dta"
 14.     forvalues g = 1/6 {
 15.         scalar th`g' = 2 * bth`g'
 16.     }
 17.     sim_fo "$out\rob_`sev'_thx2.dta"
 18.     forvalues g = 1/6 {
 19.         scalar th`g' = 0.5 * bth`g'
 20.     }
 21.     sim_fo "$out\rob_`sev'_thx05.dta"
 22. 
.     quietly streg `sev' Agecat Cardiac Nchrnx PrevICU hsp2-hsp6, ///
>         distribution(weibull) frailty(gamma) time
 23.     forvalues g = 1/6 {
 24.         scalar sig`g' = exp(-_b[/ln_p])
 25.         scalar th`g'  = e(theta)
 26.     }
 27.     sim

The pooled and per hospital calibrations give the same results, because the hospital scales differ little. The frailty variance is what moves the estimates. When it goes from half to twice the estimated value, the PRISM estimate goes from 0.200 to 0.250 and the PIM estimate from 0.100 to 0.155. The CJE values, 0.229 and 0.125, fall inside these ranges. With $\theta = 1$ the results match the logistic score within one grid step, as the theory predicts.

The split version uses half of the data, so the instrument is weaker and the sets are much wider. The logistic score on the same half moves in the same way, so the change comes from the smaller sample, not from the calibration. The split keeps exactness, but it is not suited for the main results.

## 8. Summary

What matches the CJE paper.

1. The LS confidence sets for PRISM and PIM, to the third decimal.
2. The LS point estimate for PIM.
3. The empirical quantile rank point estimates, within one grid step.

What is close but not equal.

1. The empirical quantile rank confidence sets, which are slightly narrower than in the paper.
2. The LS point estimate for PRISM, 0.160 against 0.169.

Why the remaining differences do not come from the command.

1. The CJE paper does not give the calibration of $F_o$ or the number of permutations. Section 7.6 shows that plausible calibrations move the estimates more than the gap with the paper.
2. The checks in Section 7 show that the command is stable to the sort order of the data, to the seed with 5,000 draws, and that `fo()` reproduces a known score.

With the original details, the same calibration can be rerun through `fo()` without changing the code.